In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when qqqyou create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!mkdir -p /kaggle/working/matcha_data/


In [2]:
!mkdir -p /kaggle/working/matcha_data_text/text_train

In [2]:
!mkdir -p /kaggle/working/text

In [ ]:
import shutil
import os

folder_path = '/kaggle/working/matcha_data/wavs'

if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print("Đã xóa thư mục.")
else:
    print("Thư mục không tồn tại.")


In [3]:
!cp -r /kaggle/input/subs-cu-locmoi/kaggle/working/data/subs_add_con/* /kaggle/working/matcha_data


In [4]:
!cp -r /kaggle/input/text-culocmoi/* /kaggle/working/matcha_data_text/text_train

In [8]:
!cp -r /kaggle/input/data-ipa/* /kaggle/working/matcha_data_text/text_train

In [6]:
!cp -r /kaggle/input/data-ipa/* /kaggle/working/text

In [6]:
import os

# Đường dẫn filelist gốc
filelist_path = "/kaggle/working/matcha_data_text/text_train/audio_text_train_filelist_culocmoi.txt"

# Thư mục chứa file âm thanh trong Kaggle
base_path = "/kaggle/working/matcha_data"
3
# Đọc file gốc
with open(filelist_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Xử lý từng dòng
new_lines = []
for line in lines:
    if "|" not in line:
        continue
    audio_path, text = line.strip().split("|", 1)
    # Chuẩn hóa đường dẫn
    audio_path = (
        audio_path.replace("\\", "/")   # đổi \ -> /
        .replace("../", "")             # bỏ ../
        .replace("..\\", "")            # bỏ ..\
    )
    filename = os.path.basename(audio_path)
    full_path = os.path.join(base_path, filename)
    new_lines.append(f"{full_path}|{text}")

# ⚠️ Ghi đè lại file gốc
with open(filelist_path, "w", encoding="utf-8") as f:
    f.write("\n".join(new_lines))

print("✅ Filelist đã được chỉnh sửa trực tiếp tại:", filelist_path)


✅ Filelist đã được chỉnh sửa trực tiếp tại: /kaggle/working/matcha_data_text/text_train/audio_text_train_filelist_culocmoi.txt


In [ ]:
!apt-get update -y
!apt-get install -y espeak-ng


In [8]:
!git clone https://github.com/shivammehta25/Matcha-TTS.git



Cloning into 'Matcha-TTS'...
remote: Enumerating objects: 1081, done.
remote: Counting objects: 100% (555/555), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 1081 (delta 449), reused 356 (delta 356), pack-reused 526 (from 2)
Receiving objects: 100% (1081/1081), 64.10 MiB | 43.91 MiB/s, done.
Resolving deltas: 100% (538/538), done.


In [9]:
%cd Matcha-TTS


/kaggle/working/Matcha-TTS


In [10]:
!cp -r /kaggle/input/dataaa/dataa/* /kaggle/working/Matcha-TTS/matcha/text

In [11]:
!cp -r /kaggle/input/log-culocmoi-160/* /kaggle/working/Matcha-TTS

In [12]:
import re

path = "requirements.txt"
txt = open(path).read().splitlines()
txt = [t for t in txt if not any(x in t for x in ["torchvision", "piper_phonemize"])]
txt += ["underthesea", "num2words"]
open(path, "w").write("\n".join(txt))
print("✅ Updated requirements.txt")


✅ Updated requirements.txt


In [13]:
!sed -i 's/english_cleaners2/basic_cleaners_phothong/g' matcha/cli.py


In [ ]:
!pip install -r requirements.txt


In [ ]:
!pip install -e . --find-links=https://download.pytorch.org/whl/torch_stable.html


In [18]:
from matcha.text import cleaners
from matcha.text import symbols

print("🔤 Tổng số ký hiệu trong tokenizer:", len(symbols))
print(symbols[:50])  # In thử 50 ký hiệu đầu


🔤 Tổng số ký hiệu trong tokenizer: 457
['_', '#', ';', ':', ',', '.', '!', '?', '¡', '¿', '-', '—', '…', "'", '"', '«', '»', '“', '”', '(', ')', '[', ']', '/', '%', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x']


In [21]:
import os

base = "/kaggle/working/Matcha-TTS/configs"
os.makedirs(f"{base}/data", exist_ok=True)
os.makedirs(f"{base}/experiment", exist_ok=True)

data_yaml = """_target_: matcha.data.text_mel_datamodule.TextMelDataModule
name: matcha_vi

train_filelist_path: /kaggle/working/matcha_data_text/text_train/audio_text_train_filelist_culocmoi.txt
valid_filelist_path: /kaggle/working/matcha_data_text/text_train/audio_text_val_filelist_culocmoi.txt

batch_size: 16       # mỗi GPU, tổng = 32
num_workers: 4
pin_memory: True
load_durations: false
cleaners: [basic_cleaners_phothong]
add_blank: True
n_spks: 1
n_fft: 1024
n_feats: 80
sample_rate: 22050
hop_length: 256
win_length: 1024
f_min: 0
f_max: 8000

data_statistics:
  mel_mean: -4.999980926513672
  mel_std: 2.181095838546753

seed: ${seed}
"""

exp_yaml = """# @package _global_
phonemizer:
  language: "vi"
defaults:
  - override /data: matcha_vi.yaml

tags: ["matcha_vi"]
run_name: matcha_vi

model:
  n_vocab: 500 

trainer:
  max_epochs: 220 
  check_val_every_n_epoch: 5
  devices: [0,1]        # Dùng cả 2 GPU T4
  accelerator: gpu
  strategy: ddp          # Distributed Data Parallel

test: False

extras:
  ignore_warnings: true
  enforce_tags: false
  print_config: false

hydra:
  run:
    dir: ${paths.log_dir}/${run_name}

callbacks:
  model_summary:
    max_depth: 1
  model_checkpoint:
    every_n_epochs: 20

logger:
  tensorboard:
    version: mel80band
    default_hp_metric: false

"""

with open(f"{base}/data/matcha_vi.yaml", "w") as f:
    f.write(data_yaml)
with open(f"{base}/experiment/matcha_vi.yaml", "w") as f:
    f.write(exp_yaml)

print("✅ Created configs:")
print("- configs/data/matcha_vi.yaml")
print("- configs/experiment/matcha_vi.yaml")


✅ Created configs:
- configs/data/matcha_vi.yaml
- configs/experiment/matcha_vi.yaml


In [16]:
!python matcha/utils/generate_data_statistics.py -i matcha_vi.yaml


{'mel_mean': -4.999980926513672, 'mel_std': 2.181095838546753}                  


In [ ]:
from phonemizer.backend import EspeakBackend
EspeakBackend.is_available()
espeak = EspeakBackend("vi")
print(espeak.phonemize(["xin", "chào", "mọi", "người"]))


In [ ]:
from matcha.text import symbols, _symbol_to_id

bad_lines = []
with open("/kaggle/working/matcha_data_text/text_train/audio_text_train_filelist.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            path, text = line.strip().split("|")
        except ValueError:
            print(f"❌ Lỗi format dòng {i}: {line}")
            continue

        for c in text:
            if c not in symbols:
                bad_lines.append((i, c, text))
                break

print("Số dòng có ký tự lạ:", len(bad_lines))
for i, c, text in bad_lines[:10]:
    print(f"⚠️ Dòng {i}: '{c}' không có trong symbols → {text}")


In [17]:
import os

# Đường dẫn file train
train_script_path = "/kaggle/working/Matcha-TTS/matcha/train.py"

# Nội dung vá lỗi MẠNH: Ghi đè luôn torch.load
nuclear_patch_code = """
# --- PATCH START: DISABLE PYTORCH 2.6 SECURITY CHECK (NUCLEAR OPTION) ---
import torch
import typing
import pathlib
import argparse
from omegaconf import DictConfig, ListConfig
from omegaconf.base import ContainerMetadata

# Cách 1: Thêm safe_globals (dự phòng)
try:
    torch.serialization.add_safe_globals([
        DictConfig, ListConfig, ContainerMetadata, 
        typing.Any, pathlib.PosixPath, argparse.Namespace
    ])
except:
    pass

# Cách 2: Ghi đè torch.load để luôn tắt weights_only (Giải pháp triệt để)
_original_torch_load = torch.load

def patched_torch_load(*args, **kwargs):
    # Ép buộc weights_only = False bất kể ai gọi nó
    # Điều này cho phép load checkpoint cũ thoải mái
    if 'weights_only' in kwargs:
        kwargs['weights_only'] = False
    else:
        kwargs['weights_only'] = False
        
    return _original_torch_load(*args, **kwargs)

torch.load = patched_torch_load
print(">>> [INFO] Đã ghi đè torch.load: Vô hiệu hóa hoàn toàn weights_only=True")
# --- PATCH END ---

"""

if os.path.exists(train_script_path):
    # Đọc nội dung gốc
    with open(train_script_path, "r") as f:
        original_content = f.read()
    
    # Xóa các bản vá cũ (nếu có) để tránh lặp
    # Tuy nhiên đơn giản nhất là cứ chèn đè lên đầu, Python chạy cái mới nhất là được.
    
    # Ghi đè nội dung mới lên đầu file
    new_content = nuclear_patch_code + original_content
    
    with open(train_script_path, "w") as f:
        f.write(new_content)
    
    print(f"✅ Đã áp dụng bản vá TRIỆT ĐỂ cho file: {train_script_path}")
    print("👉 Code này đã tắt hoàn toàn kiểm tra bảo mật. Bạn có thể Resume Train ngay lập tức!")
else:
    print(f"❌ Không tìm thấy file: {train_script_path}")

✅ Đã áp dụng bản vá TRIỆT ĐỂ cho file: /kaggle/working/Matcha-TTS/matcha/train.py
👉 Code này đã tắt hoàn toàn kiểm tra bảo mật. Bạn có thể Resume Train ngay lập tức!


In [ ]:
!pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# (Lưu ý: Thay cu121 bằng phiên bản cuda phù hợp với máy bạn)

In [ ]:
!python matcha/train.py experiment=matcha_vi ckpt_path=checkpoint_epoch159_culocmoi.ckpt

In [ ]:
!python matcha/train.py experiment=matcha_vi

In [20]:
import shutil

# Nén thư mục thành file zip
shutil.make_archive('/kaggle/working/Matcha-TTS/logs_culocmoi_160', 'zip', '/kaggle/working/Matcha-TTS/logs')


'/kaggle/working/Matcha-TTS/logs_culocmoi_160.zip'

In [ ]:
import os
import shutil

# 1. Dọn dẹp rác gây xung đột (Quan trọng)
if os.path.exists("speechmos"): 
    print("🧹 Đang xóa folder speechmos rác...")
    shutil.rmtree("speechmos")

# 2. Cài đặt lại thư viện sạch
!pip uninstall -y speechmos torchaudio torch # Gỡ bản cũ
!pip install -q openai-whisper jiwer librosa fastdtw scipy numpy pandas textgrid
# Cài torch bản ổn định nhất cho Kaggle
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118

print("✅ Môi trường đã sạch sẽ!")